In [1]:
import pydicom

/Users/rumethsandinu/Documents/IRP/NeoBreath/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# view dcm properties

IMAGE_PATH = "../Datasets/TCGA-LUAD/manifest-1Rd7jPNd5199284876140322680/TCGA-LUAD/TCGA-17-Z011/06-30-1982-NA-NA-32248/2.000000-NA-43644/1-1.dcm"

properties = pydicom.dcmread(IMAGE_PATH)
print(properties)

Dataset.file_meta -------------------------------
(0002, 0000) File Meta Information Group Length  UL: 194
(0002, 0001) File Meta Information Version       OB: b'\x00\x01'
(0002, 0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002, 0003) Media Storage SOP Instance UID      UI: 1.3.6.1.4.1.14519.5.2.1.7777.9002.900120867624429648234080309634
(0002, 0010) Transfer Syntax UID                 UI: Implicit VR Little Endian
(0002, 0012) Implementation Class UID            UI: 1.2.40.0.13.1.1.1
(0002, 0013) Implementation Version Name         SH: 'dcm4che-1.4.31'
-------------------------------------------------
(0008, 0000) Group Length                        UL: 426
(0008, 0005) Specific Character Set              CS: 'ISO_IR 100'
(0008, 0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'LOCALIZER']
(0008, 0016) SOP Class UID                       UI: CT Image Storage
(0008, 0018) SOP Instance UID                    UI: 1.3.6.1.4.1.14519.5.2.1.7777.9002.

In [3]:
import os
import pydicom
from collections import defaultdict
from tqdm import tqdm


In [4]:
# dataset path
DATASET_PATH = "../Datasets/TCGA-LUAD/manifest-1Rd7jPNd5199284876140322680/TCGA-LUAD"

sop_counts = defaultdict(int)

# collect all DICOM files
all_dcm_files = [os.path.join(root, f)
                 for root, _, files in os.walk(DATASET_PATH)
                 for f in files if f.endswith(".dcm")]

for filepath in tqdm(all_dcm_files, desc="Scanning DICOM files"):

    try:
        dcm = pydicom.dcmread(filepath, stop_before_pixels=True)
        sop_uid = getattr(dcm.file_meta, "MediaStorageSOPClassUID", None)
        if sop_uid:
            sop_counts[sop_uid.name] += 1  # human-readable name
    except Exception as e:
        print(f"Error reading {filepath}: {e}")

#  summary
print("\n===== SOP Class UID Distribution =====\n")
for sop_name, count in sop_counts.items():
    print(f"{sop_name}: {count} files")

Scanning DICOM files: 100%|██████████| 48931/48931 [00:41<00:00, 1186.25it/s]


===== SOP Class UID Distribution =====

CT Image Storage: 35289 files
Positron Emission Tomography Image Storage: 13058 files
Nuclear Medicine Image Storage: 6 files
Secondary Capture Image Storage: 578 files


In [5]:
img_type_counts = defaultdict(int)

# collect all DICOM files
all_dcm_files = [os.path.join(root, f)
                 for root, _, files in os.walk(DATASET_PATH)
                 for f in files if f.endswith(".dcm")]

for filepath in tqdm(all_dcm_files, desc="Scanning DICOM files"):
    try:
        dcm = pydicom.dcmread(filepath, stop_before_pixels=True)

        # extract Image Type (0008,0008)
        img_type = getattr(dcm, "ImageType", None)
        if img_type:

            # convert to tuple so it's hashable
            img_type_counts[tuple(img_type)] += 1
    except Exception as e:
        print(f"Error reading {filepath}: {e}")

# summary
print("\n===== Image Type Distribution =====\n")
for img_type, count in sorted(img_type_counts.items(), key=lambda x: -x[1]):
    print(f"{list(img_type)}: {count} files")

Scanning DICOM files: 100%|██████████| 48931/48931 [00:40<00:00, 1198.65it/s]


===== Image Type Distribution =====

['ORIGINAL', 'PRIMARY', 'AXIAL']: 18766 files
['ORIGINAL', 'PRIMARY', 'AXIAL', 'CT_SOM5 SPI']: 13158 files
['ORIGINAL', 'PRIMARY']: 12543 files
['ORIGINAL', 'PRIMARY', 'AXIAL', 'CT_SOM4 SPI']: 1043 files
['DERIVED', 'SECONDARY', 'REFORMATTED', 'MIP']: 908 files
['DERIVED', 'SECONDARY', 'OTHER', 'CSA FUSED MPR']: 422 files
['DERIVED', 'SECONDARY', 'REFORMATTED', 'AVERAGE']: 345 files
['DERIVED', 'PRIMARY', 'AXIAL', 'CT_SOM5 SPI']: 329 files
['DERIVED', 'SECONDARY', 'OTHER', 'CSA MIP', 'CSAMANIPULATED']: 224 files
['DERIVED', 'PRIMARY', 'AXIAL']: 193 files
['ORIGINAL', 'SECONDARY']: 128 files
['DERIVED', 'SECONDARY', 'OTHER', 'CSA MPR THICK', '', 'AXIAL', 'CT_SOM5 SPI']: 126 files
['ORIGINAL', 'PRIMARY', 'LOCALIZER', 'CT_SOM5 TOP']: 111 files
['ORIGINAL', 'PRIMARY', 'LOCALIZER']: 94 files
['ORIGINAL', 'PRIMARY', 'AXIAL', 'CT_SOM4 SCAN']: 94 files
['DERIVED', 'PRIMARY', 'AXIAL', 'CT_SOM5 SPO']: 73 files
['ORIGINAL', 'PRIMARY', 'AXIAL', 'CT_SOM5 SEQ']:

In [6]:
# counters
slice_counts = {"PET": 0, "CT": 0}
patient_counts = {"PET": set(), "CT": set()}

# valid SOP classes
SOP_UIDS = {
    "PET": "1.2.840.10008.5.1.4.1.1.128",
    "CT": "1.2.840.10008.5.1.4.1.1.2"
}

def is_valid_image(dcm):
    """Return True if DICOM is ORIGINAL + PRIMARY and not derived/reformatted"""
    if not hasattr(dcm, "ImageType"):
        return False
    
    img_type = [s.upper() for s in dcm.ImageType]
    if "ORIGINAL" not in img_type or "PRIMARY" not in img_type:
        return False
    
    bad_keywords = ["DERIVED", "SECONDARY", "LOCALIZER", "QC", "MONO", "MPR"]
    if any(bad in img_type for bad in bad_keywords):
        return False
    
    return True

# walk through dataset
for root, _, files in os.walk(DATASET_PATH):
    for f in files:
        if not f.endswith(".dcm"):
            continue
        
        filepath = os.path.join(root, f)
        try:
            dcm = pydicom.dcmread(filepath, stop_before_pixels=True)
            modality = getattr(dcm, "Modality", "")
            sop_uid = str(getattr(dcm, "SOPClassUID", ""))
            patient_id = str(dcm.PatientID)

            if modality == "PT" and sop_uid == SOP_UIDS["PET"] and is_valid_image(dcm):
                slice_counts["PET"] += 1
                patient_counts["PET"].add(patient_id)

            elif modality == "CT" and sop_uid == SOP_UIDS["CT"] and is_valid_image(dcm):
                slice_counts["CT"] += 1
                patient_counts["CT"].add(patient_id)

        except Exception as e:
            print(f"Error reading {filepath}: {e}")

# summary
print(f"PET: {slice_counts['PET']} slices from {len(patient_counts['PET'])} patients")
print(f"CT: {slice_counts['CT']} slices from {len(patient_counts['CT'])} patients")


PET: 12543 slices from 21 patients
CT: 33247 slices from 67 patients


In [7]:
import shutil

In [8]:
OUTPUT_PATH = "../raw/PET/A"

# counters
slice_count = 0
patient_set = set()

# ensure output folder exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# collect all DICOM files
all_dcm_files = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATASET_PATH)
    for f in files if f.endswith(".dcm")
]

# process files
for filepath in tqdm(all_dcm_files, desc="Processing PET DICOM files"):
    try:
        dcm = pydicom.dcmread(filepath, stop_before_pixels=True)
        modality = getattr(dcm, "Modality", "")
        sop_uid = str(getattr(dcm, "SOPClassUID", ""))

        if modality == "PT" and sop_uid == SOP_UIDS["PET"] and is_valid_image(dcm):
            patient_id = str(dcm.PatientID)
            full_patient_id = f"TCGA-LUAD-{patient_id}"

            dest_dir = os.path.join(OUTPUT_PATH, full_patient_id)
            os.makedirs(dest_dir, exist_ok=True)

            dest_file = os.path.join(dest_dir, os.path.basename(filepath))
            if not os.path.exists(dest_file):
                shutil.copy2(filepath, dest_file)

            slice_count += 1
            patient_set.add(full_patient_id)

    except Exception as e:
        print(f"Error reading {filepath}: {e}")

# summary
print(f"PET: {slice_count} slices copied from {len(patient_set)} patients")

Processing PET DICOM files: 100%|██████████| 48931/48931 [00:44<00:00, 1111.13it/s]

PET: 12543 slices copied from 21 patients


In [9]:
OUTPUT_PATH = "../raw/CT/A"

# counters
slice_count = 0
patient_set = set()

# ensure output folder exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# collect all DICOM files
all_dcm_files = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATASET_PATH)
    for f in files if f.endswith(".dcm")
]

# process files
for filepath in tqdm(all_dcm_files, desc="Processing CT DICOM files"):
    try:
        dcm = pydicom.dcmread(filepath, stop_before_pixels=True)
        modality = getattr(dcm, "Modality", "")
        sop_uid = str(getattr(dcm, "SOPClassUID", ""))

        if modality == "CT" and sop_uid == SOP_UIDS["CT"] and is_valid_image(dcm):
            patient_id = str(dcm.PatientID)
            full_patient_id = f"TCGA-LUAD-{patient_id}"

            dest_dir = os.path.join(OUTPUT_PATH, full_patient_id)
            os.makedirs(dest_dir, exist_ok=True)

            dest_file = os.path.join(dest_dir, os.path.basename(filepath))
            if not os.path.exists(dest_file):
                shutil.copy2(filepath, dest_file)

            slice_count += 1
            patient_set.add(full_patient_id)

    except Exception as e:
        print(f"Error reading {filepath}: {e}")

# summary
print(f"CT: {slice_count} slices copied from {len(patient_set)} patients")

Processing CT DICOM files: 100%|██████████| 48931/48931 [00:55<00:00, 880.81it/s] 

CT: 33247 slices copied from 67 patients
